# HepSetReg: full training on Colab

Runs the real training/prediction pipeline (`src/hepsetreg/runs/run_regression.py`) on a Colab GPU, by calling `train(config)`/`predict(config)` directly as library functions with a plain Python dict -- no `--config path.yaml` CLI needed, since `load_config` (the thing that parses that) is only there for the HPC's `./train`/`sbatch` entry points.

**Needs a real GPU runtime.** `train()` hardcodes `precision="bf16-mixed"`, which only works correctly on GPU -- on CPU it crashes with a dtype mismatch deep in PyTorch. The cell below checks this and stops early with a clear message instead of letting you hit that crash. In Colab: `Runtime -> Change runtime type -> T4/A100/... GPU`.

## 1. Install `hepsetreg` and check the GPU

In [ ]:
!pip install -q "git+https://github.com/els285/HepSetReg.git@colab"

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. train()'s precision='bf16-mixed' only works on GPU -- "
    "go to Runtime -> Change runtime type and pick a GPU, then re-run this cell."
)
print("GPU OK:", torch.cuda.get_device_name(0))

## 2. (Recommended) Mount Google Drive for checkpoints

Colab's local disk is ephemeral -- a disconnect (idle timeout, ~90 min on free tier; see the HepSetReg/Colab discussion in chat) loses anything not saved elsewhere. `train()` hard-codes its log directory as `tb_logs/` relative to the current directory, so rather than changing that, this just makes `tb_logs` itself a symlink into Drive -- every checkpoint/log write goes straight there, live, with no extra copy step.

Set `USE_DRIVE = False` below to skip this and keep everything local (fine for a short experiment you don't mind losing).

In [ ]:
import os

USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_LOG_DIR = "/content/drive/MyDrive/hepsetreg_runs"
    os.makedirs(DRIVE_LOG_DIR, exist_ok=True)
    if not os.path.exists("tb_logs"):
        os.symlink(DRIVE_LOG_DIR, "tb_logs")
    print("tb_logs ->", os.path.realpath("tb_logs"))

## 3. Download the data

The full train split this time (~1.3GB, so this takes a few minutes), plus val and test. Same source `setup_bsc` downloads on the HPC cluster.

In [ ]:
os.makedirs("datasets", exist_ok=True)
FOLDER_SHARE = "https://cernbox.cern.ch/s/Pd5y969ajhIi4b3/download?files="
FILES = {
    "train": "datasets/ttbar-2L_train_051026.parquet",
    "val": "datasets/ttbar-2L_val_051026.parquet",
    "test": "datasets/ttbar-2L_test_051026.parquet",
}

for split, path in FILES.items():
    remote_name = os.path.basename(path)
    if not os.path.exists(path):
        print(f"downloading {split} ({remote_name}) ...")
        !curl -L -o {path} "{FOLDER_SHARE}{remote_name}"
    else:
        print(f"{path} already downloaded, skipping.")

## 4. Features

Same field names `configs/run_regression.yaml` uses on the HPC branches.

In [ ]:
features = {
    "global": ["global_njet", "global_nelectron", "global_nmuon", "global_nbtagged", "global_met_met", "global_met_phi"],
    "jet": ["jet_pt", "jet_eta", "jet_phi", "jet_e", "jet_m", "jet_px", "jet_py", "jet_pz", "jet_bTag"],
    "lepton": ["pt", "eta", "phi", "e", "m", "px", "py", "pz", "charge", "is_electron"],
    "target": ["ttbar_mass"],
}

## 5. Train

The same defaults as `configs/run_regression.yaml`'s `train:` section on the HPC branches. `train()` picks up the GPU automatically via `accelerator="gpu"` -- PyTorch Lightning's `Trainer` handles all the device placement internally, so (unlike the `starter.ipynb` notebook) there's no manual `.to(device)` needed here.

In [ ]:
from hepsetreg.runs.run_regression import train

train_config = {
    "features": features,
    "train_path": FILES["train"],
    "val_path": FILES["val"],
    "batch_size": 4096,
    "d_model": 128,
    "nhead": 8,
    "num_layers": 6,
    "max_epochs": 50,
    "learning_rate": 1.0e-3,
    "lr_patience": 2,
    "early_stop_patience": 5,
    "accelerator": "gpu",
    "compile_model": True,
    "shape_metric": None,  # or "emd"/"mmd"/"kl"/"mmd_joint"/"sliced_emd" -- see shape_losses.py
    "shape_weight": 0.1,
}

train(train_config)  # prints "done training. checkpoint + stats saved under tb_logs/event_regression/version_N"

## 6. Predict

Fill in the `version_N` and checkpoint filename `train()` just printed above.

In [ ]:
from hepsetreg.runs.run_regression import predict

RUN_DIR = "tb_logs/event_regression/version_0"  # <- edit to match what train() printed
CHECKPOINT = f"{RUN_DIR}/checkpoints/REPLACE_ME.ckpt"  # <- edit: the one file under RUN_DIR/checkpoints/

predict_config = {
    "checkpoint": CHECKPOINT,
    "stats": f"{RUN_DIR}/stats.pt",
    "data_path": FILES["test"],
    "batch_size": 4096,
    "accelerator": "gpu",
    "output": "ttbar-2L-inference.parquet",
    "shape_metric": None,
}

predict(predict_config)

## Notes

- **Session durability**: free-tier Colab idles out in roughly 90 minutes of inactivity (keep the tab active, or use Colab Pro's background execution) and caps sessions around 12 hours regardless. With `USE_DRIVE = True` above, a disconnect mid-training still leaves whatever checkpoints were written so far safely on Drive.
- **GPU VRAM**: `d_model=128`/6 layers/`batch_size=4096` comfortably fits on any Colab GPU tier -- VRAM was never the constraint here.
- To change any setting, just edit `train_config`/`predict_config` directly -- there's no YAML file involved in this notebook.